## Занятие 8. Визуализация связей (Scatter plots и парные графики)

Анализ данных `marketing_data.csv`: рекламные бюджеты, посещения, конверсии и продажи. Корреляция описывает связь, но не доказывает причинно-следственную зависимость.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
df = pd.read_csv('marketing_data.csv')
numeric_cols = ['TV_Spend', 'Radio_Spend', 'Social_Spend', 'Visits', 'Conversions', 'Sales']
print('Размер данных:', df.shape)
print('Пропуски:', df.isna().sum().to_dict())
df.head()

### Задание 1. Scatter plot: TV-бюджет и продажи

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.regplot(data=df, x='TV_Spend', y='Sales', ci=95, scatter_kws={'alpha': 0.55}, ax=axes[0], label='Линейный тренд')
sns.regplot(data=df, x='TV_Spend', y='Sales', order=2, ci=None, scatter=False, color='darkorange', ax=axes[0], label='Квадратичный тренд')
axes[0].set_title('TV-бюджет и продажи: линейный и квадратичный тренды')
axes[0].legend()
hb = axes[1].hexbin(df['TV_Spend'], df['Sales'], gridsize=22, cmap='Blues', mincnt=1)
fig.colorbar(hb, ax=axes[1], label='Количество наблюдений')
axes[1].set(xlabel='TV_Spend', ylabel='Sales', title='Плотность наблюдений (hexbin)')
plt.tight_layout()
plt.show()
coef = np.polyfit(df['TV_Spend'], df['Sales'], 2)
print(f'Квадратичный тренд: Sales ≈ {coef[0]:.3f}·TV² + {coef[1]:.3f}·TV + {coef[2]:.3f}')
print(f'Вершина параболы: TV ≈ {-coef[1]/(2*coef[0]):.1f}')

**Вывод:** наблюдается положительная связь TV-бюджета с продажами, но разброс велик. Отрицательный квадратичный коэффициент указывает на возможное замедление роста при больших бюджетах (вершина около 43); это не доказательство, что увеличение бюджета само по себе снижает продажи: есть другие факторы и выбросы. Hexbin показывает, где сосредоточены наблюдения.

### Задание 2. Категории продуктов и размер точек по посещениям

In [ ]:
plt.figure(figsize=(11, 6))
sns.scatterplot(data=df, x='TV_Spend', y='Sales', hue='Product_Category', size='Visits', sizes=(25, 250), alpha=0.7)
plt.title('Продажи и TV-бюджет по категориям (размер = Visits)')
plt.tight_layout()
plt.show()
print('Продажи по категориям:')
print(df.groupby('Product_Category')['Sales'].agg(['count', 'mean', 'median']).round(2))
print('Медианы продаж в квартилях TV-бюджета:')
print(df.assign(TV_группа=pd.qcut(df['TV_Spend'], 4)).groupby(['TV_группа', 'Product_Category'], observed=True)['Sales'].median().unstack().round(1))

**Вывод:** крупные точки (больше посещений) чаще встречаются при более высоких продажах. У категорий есть перекрытие при одинаковом TV-бюджете: устойчивого лидера для всех уровней бюджета нет. Сравнение медиан внутри квартилей TV помогает не спутать эффект категории с различиями в бюджетах.

### Задание 3. Матрица парных графиков

In [ ]:
grid = sns.pairplot(df, vars=numeric_cols, hue='Product_Category', diag_kind='kde', plot_kws={'alpha': 0.5, 's': 20}, diag_kws={'fill': False})
grid.fig.suptitle('Попарные связи числовых признаков', y=1.02)
plt.show()
print('Самые сильные парные корреляции:')
corr = df[numeric_cols].corr()
pairs = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool)).stack().sort_values(ascending=False)
print(pairs.head(6).round(3))

**Вывод:** наиболее заметны связи TV_Spend–Visits (r ≈ 0.87), Conversions–Sales (r ≈ 0.76) и Visits–Conversions (r ≈ 0.62). Social_Spend почти не коррелирует с остальными признаками. Категории на матрице заметно перекрываются; отдельные продажи до 400 и 317 выделяются как выбросы.

### Задание 4. Joint plot и логарифмическое преобразование

In [ ]:
g = sns.jointplot(data=df, x='TV_Spend', y='Sales', kind='reg', height=6, scatter_kws={'alpha': 0.5})
g.fig.suptitle('Исходные TV-бюджет и продажи', y=1.02)
plt.show()
log_df = df[['TV_Spend', 'Sales']].apply(np.log1p).rename(columns={'TV_Spend': 'log1p_TV_Spend', 'Sales': 'log1p_Sales'})
g = sns.jointplot(data=log_df, x='log1p_TV_Spend', y='log1p_Sales', kind='reg', height=6, scatter_kws={'alpha': 0.5})
g.fig.suptitle('Логарифмы TV-бюджета и продаж', y=1.02)
plt.show()
print(f'Корреляция Пирсона до преобразования: {df.TV_Spend.corr(df.Sales):.3f}')
print(f'Корреляция Пирсона после преобразования: {log_df.log1p_TV_Spend.corr(log_df.log1p_Sales):.3f}')
print('Асимметрия до:', df[['TV_Spend', 'Sales']].skew().round(2).to_dict())
print('Асимметрия после:', log_df.skew().round(2).to_dict())

**Вывод:** коэффициент Пирсона возрастает примерно с 0.535 до 0.683; на логарифмической шкале связь выглядит более ровной, но преобразование не доказывает линейную причинную зависимость. Асимметрия Sales уменьшается с 2.56 до −0.73; для TV_Spend, наоборот, модуль асимметрии увеличивается с 0.09 до 1.10. Следовательно, обе переменные одновременно *не* стали более симметричными.

### Задание 5. Тепловая карта корреляций

In [ ]:
corr = df[numeric_cols].corr()
plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, vmin=-1, vmax=1, square=True)
plt.title('Корреляционная матрица числовых признаков')
plt.tight_layout()
plt.show()
print('Связь признаков с продажами:')
print(corr['Sales'].drop('Sales').sort_values(ascending=False).round(3))
print('Корреляции рекламных бюджетов:')
print(corr.loc[['TV_Spend', 'Radio_Spend', 'Social_Spend'], ['TV_Spend', 'Radio_Spend', 'Social_Spend']].round(3))

**Вывод:** сильнее всего с Sales связаны Conversions (r ≈ 0.76), затем Visits (r ≈ 0.59). Среди рекламных каналов лидирует TV (r ≈ 0.53), далее Radio (r ≈ 0.39), Social почти не связан с продажами (r ≈ 0.03). TV и Radio умеренно коррелируют (r ≈ 0.46), выраженной парной мультиколлинеарности бюджетов не видно. Для дальнейшей проверки эффективности стоит сначала изучить TV, затем Radio, но перераспределять бюджеты лишь на основании корреляций нельзя: нужны учёт затрат, других факторов и экспериментальная проверка.